<a href="https://colab.research.google.com/github/madalamanikanta/ImageCaptioning_MiniProject/blob/manikanta-dev/06_Dataset_Split_and_Tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 06 — Dataset Split & Tokenization


## Step 1 — Connect Google Drive

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## Step 2 — Import Required Libraries


In [2]:
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

## Step 3 — Define Project Paths


In [3]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/ImageCaptioning_MiniProject"
)
PROCESSED_DIR = PROJECT_DIR / "Processed"
cleaned_file = PROCESSED_DIR / "cleaned_dataset.csv"
print("Project directory :", PROJECT_DIR)
print("Cleaned dataset   :", cleaned_file)

Project directory : /content/drive/MyDrive/ImageCaptioning_MiniProject
Cleaned dataset   : /content/drive/MyDrive/ImageCaptioning_MiniProject/Processed/cleaned_dataset.csv


## Step 4 — Load Cleaned Dataset


In [4]:
df = pd.read_csv(cleaned_file)
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (599278, 3)


,image_path,caption,dataset
0,/content/drive/MyDrive/ImageCaptioning_MiniPro...,<start> A child in a pink dress is climbing up...,Flickr8K
1,/content/drive/MyDrive/ImageCaptioning_MiniPro...,<start> A girl going into a wooden building <end>,Flickr8K
2,/content/drive/MyDrive/ImageCaptioning_MiniPro...,<start> A little girl climbing into a wooden p...,Flickr8K
3,/content/drive/MyDrive/ImageCaptioning_MiniPro...,<start> A little girl climbing the stairs to h...,Flickr8K
4,/content/drive/MyDrive/ImageCaptioning_MiniPro...,<start> A little girl in a pink dress going in...,Flickr8K


## Step 5 — Inspect Dataset


In [5]:
print("Columns:")
print(df.columns.tolist())

print()

print("Total caption rows :", len(df))

print(
    "Unique images      :",
    df["image_path"].nunique()
)

print()

print("Captions per dataset:")
print(df["dataset"].value_counts())

Columns:
['image_path', 'caption', 'dataset']

Total caption rows : 599278
Unique images      : 119865

Captions per dataset:
dataset
COCO2017     400016
Flickr30K    158848
Flickr8K      40414
Name: count, dtype: int64


## Step 6 — Create Unique Image List


In [6]:
unique_images = (
    df["image_path"]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Unique images:", len(unique_images))

Unique images: 119865


## Step 7 — Create Training and Temporary Splits


In [7]:
train_images, temp_images = train_test_split(
    unique_images,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

print("Training images :", len(train_images))
print("Temporary images:", len(temp_images))

Training images : 95892
Temporary images: 23973


## Step 8 — Create Validation and Test Splits


In [8]:
val_images, test_images = train_test_split(
    temp_images,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

In [9]:
print("Training images   :", len(train_images))
print("Validation images :", len(val_images))
print("Test images       :", len(test_images))

Training images   : 95892
Validation images : 11986
Test images       : 11987


## Step 9 — Verify No Image Leakage


In [11]:
train_set = set(train_images)
val_set = set(val_images)
test_set = set(test_images)

In [12]:
print("Train ∩ Validation:", len(train_set & val_set))
print("Train ∩ Test      :", len(train_set & test_set))
print("Validation ∩ Test :", len(val_set & test_set))

Train ∩ Validation: 0
Train ∩ Test      : 0
Validation ∩ Test : 0


## Step 10 — Assign Captions to Their Image Split


In [13]:
train_df = df[df["image_path"].isin(train_set)].copy()
val_df = df[df["image_path"].isin(val_set)].copy()
test_df = df[df["image_path"].isin(test_set)].copy()

In [14]:
print("Train rows :", len(train_df))
print("Val rows   :", len(val_df))
print("Test rows  :", len(test_df))

Train rows : 479416
Val rows   : 59923
Test rows  : 59939


## Step 11 — Verify Final Dataset Splits


In [16]:
print("=" * 60)
print("DATASET SPLIT SUMMARY")
print("=" * 60)
print()
print("TRAIN")
print("Images   :", train_df["image_path"].nunique())
print("Captions :", len(train_df))

print()

print("VALIDATION")
print("Images   :", val_df["image_path"].nunique())
print("Captions :", len(val_df))

print()

print("TEST")
print("Images   :", test_df["image_path"].nunique())
print("Captions :", len(test_df))

print()

print("TOTAL UNIQUE IMAGES :",
      train_df["image_path"].nunique()
      + val_df["image_path"].nunique()
      + test_df["image_path"].nunique())

print("ORIGINAL UNIQUE IMAGES:",
      df["image_path"].nunique())

DATASET SPLIT SUMMARY

TRAIN
Images   : 95892
Captions : 479416

VALIDATION
Images   : 11986
Captions : 59923

TEST
Images   : 11987
Captions : 59939

TOTAL UNIQUE IMAGES : 119865
ORIGINAL UNIQUE IMAGES: 119865


## Step 12 — Save Dataset Splits


In [17]:
train_file = PROCESSED_DIR / "train_dataset.csv"
val_file = PROCESSED_DIR / "val_dataset.csv"
test_file = PROCESSED_DIR / "test_dataset.csv"
train_df.to_csv(train_file, index=False)
val_df.to_csv(val_file, index=False)
test_df.to_csv(test_file, index=False)

In [18]:
print("Saved:")
print(train_file)
print(val_file)
print(test_file)

Saved:
/content/drive/MyDrive/ImageCaptioning_MiniProject/Processed/train_dataset.csv
/content/drive/MyDrive/ImageCaptioning_MiniProject/Processed/val_dataset.csv
/content/drive/MyDrive/ImageCaptioning_MiniProject/Processed/test_dataset.csv


## Step 13 — Import Tokenization Utilities

In [19]:
from collections import Counter
import json
import re
import numpy as np

## Step 14 — Verify Caption Format

In [20]:
print("Sample captions:")
print()

for caption in train_df["caption"].head(5):
    print(caption)

print()
print("Starts with <start>:",
      train_df["caption"].str.startswith("<start>").all())

print("Ends with <end>:",
      train_df["caption"].str.endswith("<end>").all())

Sample captions:

<start> A black dog and a spotted dog are fighting <end>
<start> A black dog and a tricolored dog playing with each other on the road <end>
<start> A black dog and a white dog with brown spots are staring at each other in the street <end>
<start> Two dogs of different breeds looking at each other on the road <end>
<start> Two dogs on pavement moving toward each other <end>

Starts with <start>: True
Ends with <end>: True


##Step 15 — Build Vocabulary from Training Captions

In [21]:
print("=" * 60)
print("BUILDING VOCABULARY")
print("=" * 60)

word_counter = Counter()

for caption in train_df["caption"]:
    tokens = caption.split()
    word_counter.update(tokens)

print("Unique tokens before filtering:",
      len(word_counter))

BUILDING VOCABULARY
Unique tokens before filtering: 36574


##Step 16 — Create Vocabulary

In [22]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
START_TOKEN = "<start>"
END_TOKEN = "<end>"

PAD_ID = 0
UNK_ID = 1
START_ID = 2
END_ID = 3

MIN_WORD_FREQUENCY = 2

word_to_id = {
    PAD_TOKEN: PAD_ID,
    UNK_TOKEN: UNK_ID,
    START_TOKEN: START_ID,
    END_TOKEN: END_ID
}

for word, count in word_counter.most_common():
    if count >= MIN_WORD_FREQUENCY:
        if word not in word_to_id:
            word_to_id[word] = len(word_to_id)

id_to_word = {
    idx: word
    for word, idx in word_to_id.items()
}

print("Vocabulary size:", len(word_to_id))

Vocabulary size: 21649


##Step 17 — Vocabulary Statistics

In [23]:
print("=" * 60)
print("VOCABULARY STATISTICS")
print("=" * 60)

print("Vocabulary size :", len(word_to_id))
print("PAD ID          :", PAD_ID)
print("UNK ID           :", UNK_ID)
print("START ID         :", START_ID)
print("END ID           :", END_ID)

print()
print("Most common words:")

for word, count in word_counter.most_common(20):
    print(f"{word:20s} {count}")

VOCABULARY STATISTICS
Vocabulary size : 21649
PAD ID          : 0
UNK ID           : 1
START ID         : 2
END ID           : 3

Most common words:
a                    520478
<start>              479416
<end>                479416
A                    276273
in                   180600
on                   160808
the                  150881
of                   145899
with                 118322
and                  117646
is                   92381
man                  75344
to                   53248
are                  40751
at                   37993
sitting              37017
woman                36828
people               32839
white                32691
Two                  32111


##Step 18 — Calculate Maximum Caption Length

In [24]:
caption_lengths = train_df["caption"].apply(
    lambda x: len(x.split())
)

MAX_LENGTH = int(caption_lengths.max())

print("=" * 60)
print("CAPTION LENGTH STATISTICS")
print("=" * 60)

print("Minimum length :", caption_lengths.min())
print("Maximum length :", caption_lengths.max())
print("Average length :", round(caption_lengths.mean(), 2))
print("Median length  :", caption_lengths.median())

CAPTION LENGTH STATISTICS
Minimum length : 5
Maximum length : 80
Average length : 12.96
Median length  : 12.0


##Step 19 — Convert Caption to Token IDs

In [25]:
def caption_to_ids(caption):
    tokens = caption.split()

    return [
        word_to_id.get(token, UNK_ID)
        for token in tokens
    ]

In [26]:
sample_caption = train_df["caption"].iloc[0]

sample_ids = caption_to_ids(sample_caption)

print("Caption:")
print(sample_caption)

print()
print("Token IDs:")
print(sample_ids)

Caption:
<start> A black dog and a spotted dog are fighting <end>

Token IDs:
[2, 5, 27, 26, 11, 4, 2037, 26, 15, 1052, 3]


##Step 20 — Verify Tokenization

In [27]:
decoded_caption = [
    id_to_word.get(idx, UNK_TOKEN)
    for idx in sample_ids
]

print("Original:")
print(sample_caption)

print()
print("Decoded:")
print(" ".join(decoded_caption))

Original:
<start> A black dog and a spotted dog are fighting <end>

Decoded:
<start> A black dog and a spotted dog are fighting <end>


##Step 21 — Verify Special Tokens

In [28]:
print("First token ID :", sample_ids[0])
print("Last token ID  :", sample_ids[-1])

print()

print("Expected START ID:", START_ID)
print("Expected END ID  :", END_ID)

First token ID : 2
Last token ID  : 3

Expected START ID: 2
Expected END ID  : 3


##Step 22 — Save Vocabulary

In [29]:
VOCAB_FILE = PROCESSED_DIR / "vocabulary.json"

vocab_data = {
    "word_to_id": word_to_id,
    "id_to_word": {
        str(k): v
        for k, v in id_to_word.items()
    },
    "vocab_size": len(word_to_id),
    "max_length": MAX_LENGTH,
    "special_tokens": {
        "PAD": PAD_TOKEN,
        "UNK": UNK_TOKEN,
        "START": START_TOKEN,
        "END": END_TOKEN
    }
}

with open(VOCAB_FILE, "w", encoding="utf-8") as f:
    json.dump(vocab_data, f, ensure_ascii=False, indent=2)

print("Vocabulary saved:")
print(VOCAB_FILE)

Vocabulary saved:
/content/drive/MyDrive/ImageCaptioning_MiniProject/Processed/vocabulary.json


##Step 23 — Final Tokenization Summary

In [30]:
print("=" * 60)
print("TOKENIZATION SUMMARY")
print("=" * 60)

print("Training captions :", len(train_df))
print("Validation captions:", len(val_df))
print("Test captions     :", len(test_df))

print()
print("Vocabulary size   :", len(word_to_id))
print("Maximum length    :", MAX_LENGTH)

print()
print("Special tokens:")
print("<PAD>   :", PAD_ID)
print("<UNK>   :", UNK_ID)
print("<start> :", START_ID)
print("<end>   :", END_ID)

print()
print("Vocabulary file:")
print(VOCAB_FILE)

TOKENIZATION SUMMARY
Training captions : 479416
Validation captions: 59923
Test captions     : 59939

Vocabulary size   : 21649
Maximum length    : 80

Special tokens:
<PAD>   : 0
<UNK>   : 1
<start> : 2
<end>   : 3

Vocabulary file:
/content/drive/MyDrive/ImageCaptioning_MiniProject/Processed/vocabulary.json


# Step 24 — Create Image-to-Index Mapping



In [31]:
image_to_index = {
    str(image_path): idx
    for idx, image_path in enumerate(unique_images)
}

print("Total unique images:", len(image_to_index))

Total unique images: 119865


##Step 25 — Add Image Index to Each Split

In [32]:
# Step 25 — Map Each Caption to Its Image Feature Index

train_image_indices = train_df["image_path"].map(
    lambda x: image_to_index[str(x)]
).to_numpy(dtype=np.int32)

val_image_indices = val_df["image_path"].map(
    lambda x: image_to_index[str(x)]
).to_numpy(dtype=np.int32)

test_image_indices = test_df["image_path"].map(
    lambda x: image_to_index[str(x)]
).to_numpy(dtype=np.int32)

print("Train image indices:", train_image_indices.shape)
print("Validation image indices:", val_image_indices.shape)
print("Test image indices:", test_image_indices.shape)

Train image indices: (479416,)
Validation image indices: (59923,)
Test image indices: (59939,)


# Step 26 — Convert Captions to Padded Token Sequences

In [33]:
def captions_to_sequences(captions, max_length):
    sequences = np.full(
        (len(captions), max_length),
        PAD_ID,
        dtype=np.int32
    )

    for i, caption in enumerate(captions):
        token_ids = caption_to_ids(caption)

        length = min(len(token_ids), max_length)

        sequences[i, :length] = token_ids[:length]

    return sequences


##Step 27 — Tokenize Train / Validation / Test Captions

In [34]:
# Step 27 — Tokenize All Dataset Splits

print("Tokenizing training captions...")

train_sequences = captions_to_sequences(
    train_df["caption"].values,
    MAX_LENGTH
)

print("Tokenizing validation captions...")

val_sequences = captions_to_sequences(
    val_df["caption"].values,
    MAX_LENGTH
)

print("Tokenizing test captions...")

test_sequences = captions_to_sequences(
    test_df["caption"].values,
    MAX_LENGTH
)

print()
print("Train sequences :", train_sequences.shape)
print("Validation sequences :", val_sequences.shape)
print("Test sequences :", test_sequences.shape)

Tokenizing training captions...
Tokenizing validation captions...
Tokenizing test captions...

Train sequences : (479416, 80)
Validation sequences : (59923, 80)
Test sequences : (59939, 80)


##Step 28 — Verify Padding

In [35]:
# Step 28 — Verify Padding

sample_sequence = train_sequences[0]

print("Token IDs:")
print(sample_sequence)

print()
print("Number of PAD tokens:",
      np.sum(sample_sequence == PAD_ID))

Token IDs:
[   2    5   27   26   11    4 2037   26   15 1052    3    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0]

Number of PAD tokens: 69


##Step 29 — Decode a Tokenized Caption

In [36]:
# Step 29 — Decode Tokenized Caption

sample_sequence = train_sequences[0]

decoded_tokens = [
    id_to_word.get(int(token_id), UNK_TOKEN)
    for token_id in sample_sequence
    if token_id != PAD_ID
]

print("Original caption:")
print(train_df["caption"].iloc[0])

print()
print("Decoded caption:")
print(" ".join(decoded_tokens))

Original caption:
<start> A black dog and a spotted dog are fighting <end>

Decoded caption:
<start> A black dog and a spotted dog are fighting <end>


In [37]:
##Step 30 — Save Tokenized Data

In [38]:
# Step 30 — Save Tokenized Data

TRAIN_SEQ_FILE = PROCESSED_DIR / "train_caption_sequences.npy"
VAL_SEQ_FILE = PROCESSED_DIR / "val_caption_sequences.npy"
TEST_SEQ_FILE = PROCESSED_DIR / "test_caption_sequences.npy"

TRAIN_IMG_IDX_FILE = PROCESSED_DIR / "train_image_indices.npy"
VAL_IMG_IDX_FILE = PROCESSED_DIR / "val_image_indices.npy"
TEST_IMG_IDX_FILE = PROCESSED_DIR / "test_image_indices.npy"

np.save(TRAIN_SEQ_FILE, train_sequences)
np.save(VAL_SEQ_FILE, val_sequences)
np.save(TEST_SEQ_FILE, test_sequences)

np.save(TRAIN_IMG_IDX_FILE, train_image_indices)
np.save(VAL_IMG_IDX_FILE, val_image_indices)
np.save(TEST_IMG_IDX_FILE, test_image_indices)

print("Saved tokenized data:")
print()
print(TRAIN_SEQ_FILE)
print(VAL_SEQ_FILE)
print(TEST_SEQ_FILE)

print()
print("Saved image indices:")
print(TRAIN_IMG_IDX_FILE)
print(VAL_IMG_IDX_FILE)
print(TEST_IMG_IDX_FILE)

Saved tokenized data:

/content/drive/MyDrive/ImageCaptioning_MiniProject/Processed/train_caption_sequences.npy
/content/drive/MyDrive/ImageCaptioning_MiniProject/Processed/val_caption_sequences.npy
/content/drive/MyDrive/ImageCaptioning_MiniProject/Processed/test_caption_sequences.npy

Saved image indices:
/content/drive/MyDrive/ImageCaptioning_MiniProject/Processed/train_image_indices.npy
/content/drive/MyDrive/ImageCaptioning_MiniProject/Processed/val_image_indices.npy
/content/drive/MyDrive/ImageCaptioning_MiniProject/Processed/test_image_indices.npy


##Step 31 — Final Verification

In [39]:
# Step 31 — Final Verification

print("=" * 60)
print("TOKENIZED DATA VERIFICATION")
print("=" * 60)

print("Train captions       :", len(train_sequences))
print("Train image indices  :", len(train_image_indices))

print("Validation captions  :", len(val_sequences))
print("Validation image idx :", len(val_image_indices))

print("Test captions        :", len(test_sequences))
print("Test image indices   :", len(test_image_indices))

print()
print("Sequence length      :", train_sequences.shape[1])
print("Vocabulary size      :", len(word_to_id))

print()
print("All train aligned:",
      len(train_sequences) == len(train_image_indices))

print("All validation aligned:",
      len(val_sequences) == len(val_image_indices))

print("All test aligned:",
      len(test_sequences) == len(test_image_indices))

TOKENIZED DATA VERIFICATION
Train captions       : 479416
Train image indices  : 479416
Validation captions  : 59923
Validation image idx : 59923
Test captions        : 59939
Test image indices   : 59939

Sequence length      : 80
Vocabulary size      : 21649

All train aligned: True
All validation aligned: True
All test aligned: True
